In [1]:
import numpy as np
import random
from typing import Optional, Tuple, Dict, List


# ============================================================
# Environment
# ============================================================

class PianoEnv:

    def __init__(self,
                 length=20,
                 n_actions=10,
                 max_steps=None,
                 seed=0):

        self.length = length - 1
        self.n_states = length
        self.n_actions = n_actions

        self.rng = random.Random(seed)
        self.correct_map = [self.rng.randrange(n_actions)
                            for _ in range(length)]

        print("Correct map:", self.correct_map)

        self.max_steps = max_steps if max_steps else 2 * length

        self.allowed_actions_map = [
            list(range(n_actions)) for _ in range(length)
        ]

    def reset(self):

        self.state = 0
        self.steps = 0
        return self.state

    def step(self, action):

        self.steps += 1

        reward = -0.8
        done = False

        if action == self.correct_map[self.state]:

            reward = 1.0
            self.state += 1

            if self.state >= self.length:
                done = True

        if self.steps >= self.max_steps:
            done = True

        return self.state, reward, done, {}

    def allowed_actions(self, state):

        return self.allowed_actions_map[state]


# ============================================================
# Student agent
# ============================================================

class StudentAgent:

    def __init__(self,
                 name,
                 n_states,
                 n_actions,
                 lr=0.1,
                 gamma=0.95,
                 eps=0.1):

        self.name = name
        self.Q = np.zeros((n_states, n_actions))

        self.lr = lr
        self.gamma = gamma
        self.eps = eps

    # --------------------------------------------------------
    # Action selection
    # --------------------------------------------------------

    def act(self, env_state, allowed):

        if random.random() < self.eps:
            return random.choice(allowed)

        return allowed[np.argmax(self.Q[env_state, allowed])]

    # --------------------------------------------------------
    # Direct learning
    # --------------------------------------------------------

    def learn(self, s, a, r, s_next, done):

        best_next = np.max(self.Q[s_next])

        target = r if done else r + self.gamma * best_next

        self.Q[s, a] += self.lr * (target - self.Q[s, a])

    # --------------------------------------------------------
    # Observational learning from peer
    # --------------------------------------------------------

    def observe_peer(self,
                     s,
                     a,
                     r,
                     s_next,
                     done,
                     peer_Q,
                     obs_factor=0.4,
                     imitation_factor=0.2):

        best_next = np.max(self.Q[s_next])

        target = r if done else r + self.gamma * best_next

        # observational TD learning
        self.Q[s, a] += obs_factor * self.lr * (
            target - self.Q[s, a]
        )

        # imitation coupling
        self.Q[s] = (
            (1 - imitation_factor) * self.Q[s]
            + imitation_factor * peer_Q[s]
        )


# ============================================================
# Optional Coach
# ============================================================

class CoachAgent:

    def __init__(self,
                 n_states,
                 n_actions,
                 intervention_prob=0.1):

        self.Q = np.zeros((n_states, n_actions))

        self.intervention_prob = intervention_prob

    def should_intervene(self):

        return random.random() < self.intervention_prob

    def suggest_action(self, state, allowed):

        return allowed[np.argmax(self.Q[state, allowed])]

    def learn(self, s, a, r, s_next, done, lr=0.05, gamma=0.95):

        best_next = np.max(self.Q[s_next])

        target = r if done else r + gamma * best_next

        self.Q[s, a] += lr * (target - self.Q[s, a])


# ============================================================
# Dyadic Student–Student System with Optional Coach
# ============================================================

class StudentStudentDyad:

    def __init__(self,
                 env,
                 studentA,
                 studentB,
                 coach=None):

        self.env = env

        self.studentA = studentA
        self.studentB = studentB

        self.coach = coach

        self.turn = 0

    # --------------------------------------------------------
    # Tupled dyadic state
    # --------------------------------------------------------

    def joint_state(self, env_state):

        coach_state = env_state if self.coach else None

        return (
            env_state,
            env_state,   # student A belief
            env_state,   # student B belief
            coach_state
        )

    # --------------------------------------------------------
    # Run episode
    # --------------------------------------------------------

    def run_episode(self, max_steps=200):

        s = self.env.reset()

        total_reward = 0

        for t in range(max_steps):

            allowed = self.env.allowed_actions(s)

            acting_student = (
                self.studentA if self.turn == 0 else self.studentB
            )

            observing_student = (
                self.studentB if self.turn == 0 else self.studentA
            )

            # ------------------------------------------------
            # Coach intervention (optional)
            # ------------------------------------------------

            if self.coach and self.coach.should_intervene():

                action = self.coach.suggest_action(s, allowed)

            else:

                action = acting_student.act(s, allowed)

            # ------------------------------------------------
            # Environment step
            # ------------------------------------------------

            s_next, reward, done, _ = self.env.step(action)

            total_reward += reward

            # ------------------------------------------------
            # Acting student learns directly
            # ------------------------------------------------

            acting_student.learn(
                s, action, reward, s_next, done
            )

            # ------------------------------------------------
            # Peer learns observationally
            # ------------------------------------------------

            observing_student.observe_peer(
                s,
                action,
                reward,
                s_next,
                done,
                acting_student.Q
            )

            # ------------------------------------------------
            # Coach learns (if present)
            # ------------------------------------------------

            if self.coach:

                self.coach.learn(
                    s, action, reward, s_next, done
                )

            # switch turns
            self.turn = 1 - self.turn

            s = s_next

            if done:
                break

        return total_reward

In [2]:
env = PianoEnv(length=20, n_actions=10)

studentA = StudentAgent("A", env.n_states, env.n_actions)
studentB = StudentAgent("B", env.n_states, env.n_actions)

coach = CoachAgent(env.n_states, env.n_actions,
                   intervention_prob=0.15)

dyad = StudentStudentDyad(env,
                          studentA,
                          studentB,
                          coach)

for episode in range(500):

    reward = dyad.run_episode()

    if episode % 50 == 0:
        print("Episode", episode, "Reward:", reward)

Correct map: [6, 6, 0, 4, 8, 7, 6, 4, 7, 5, 9, 3, 8, 2, 4, 2, 1, 9, 4, 8]
Episode 0 Reward: -21.200000000000014
Episode 50 Reward: 14.199999999999996
Episode 100 Reward: 18.2
Episode 150 Reward: 17.4
Episode 200 Reward: 18.2
Episode 250 Reward: 19.0
Episode 300 Reward: 16.6
Episode 350 Reward: 19.0
Episode 400 Reward: 18.2
Episode 450 Reward: 18.2
